# Study 19 — Geolocalizzazione eventi RSS: euristica vs LLM locale

**Problema**: 0/1996 eventi `origin='rss'` hanno `location_name`/`lat`/`lon` (vedi Overview dashboard —
Cuba/Venezuela mostrano solo terremoti USGS, mai le notizie politiche). Solo gli ingestor geo-nativi
(USGS/FIRMS/PortWatch) scrivono `location_name`; nulla lo deriva per gli eventi RSS clusterizzati.

**Regola richiesta (utente, 2026-07-13)**:
- Notizia su relazione bilaterale/multilaterale tra grandi potenze (es. USA-Iran, USA-Israele) →
  **non localizzare** (non è un evento ancorato a un luogo, è una relazione tra attori).
- Notizia su un solo paese → localizzare lì.
- Notizia dove un attore agisce su un bersaglio *tramite* un terzo paese (es. "USA su Cuba via
  Venezuela") → localizzare sul **bersaglio** (Cuba), non sull'attore (USA) né sul mezzo (Venezuela).

Questo è un task di ruolo semantico (attore/bersaglio/mezzo), non deducibile in modo affidabile da
solo conteggio co-occorrenze. Questo notebook valida due approcci **prima di toccare `extract.py`**:

1. **Euristica** (zero dipendenze, istantanea) — quanto arriva lontano da sola?
2. **Qwen3 4B locale via Ollama** (`LLMClient(backend="qwen-local")`, già cablato nel progetto) — sui
   casi che l'euristica non risolve.

Nessuna scrittura sul DB in questo notebook — solo lettura + prototipo in memoria.

In [1]:
import os
import sqlite3
from pathlib import Path

notebook_dir = Path.cwd()
while not (notebook_dir / "data/db/pathosphere.db").exists():
    notebook_dir = notebook_dir.parent
    if notebook_dir == notebook_dir.parent:
        raise RuntimeError("project root not found")
os.chdir(notebook_dir)

import pandas as pd

from pathosphere.config import get_settings
from pathosphere.db.schema import get_connection

settings = get_settings()
conn = get_connection(settings.db_path)
conn.row_factory = sqlite3.Row
pd.set_option("display.max_colwidth", 120)

## 1. Baseline — copertura geolocalizzazione per origine

In [2]:
coverage = pd.read_sql_query(
    """
    SELECT origin, COUNT(*) AS tot,
           SUM(location_name IS NOT NULL) AS has_location_name,
           SUM(lat IS NOT NULL) AS has_latlon
    FROM events GROUP BY origin
    """,
    conn,
)
coverage

,origin,tot,has_location_name,has_latlon
0,firms,60,60,60
1,ioda,219,0,0
2,portwatch,4904,4904,4173
3,rss,1996,0,0
4,usgs,1062,1062,1062


Confermato: RSS (1996 eventi) e IODA (219) non hanno mai `location_name`. USGS/FIRMS/PortWatch sono
geo-nativi (coordinate nel dato grezzo) e già coperti al 100/85%. Il gap è tutto e solo sul lato
semantico (RSS).

## 2. Entità country/location per evento RSS (dato grezzo per l'euristica)

Per ogni evento, le entità `location`/`country` menzionate nei documenti del cluster, con conteggio
menzioni. Nessuna deduplicazione di ruolo qui — solo la materia prima.

In [3]:
event_locations = pd.read_sql_query(
    """
    SELECT e.id AS event_id, e.title,
           COALESCE(en.canonical_name, en.name) AS country,
           SUM(de.mentions) AS mentions
    FROM events e
    JOIN event_documents ed ON ed.event_id = e.id
    JOIN document_entities de ON de.document_id = ed.document_id
    JOIN entities en ON en.id = de.entity_id
    WHERE e.origin = 'rss' AND en.entity_type IN ('location', 'country')
    GROUP BY e.id, country
    """,
    conn,
)
print(f"{event_locations['event_id'].nunique()} eventi RSS con almeno 1 entità location/country")
event_locations.head(10)

1690 eventi RSS con almeno 1 entità location/country


,event_id,title,country,mentions
0,121951,Canadian policeman killed during investigation tied to US consulate shooting,Canada,2
1,121951,Canadian policeman killed during investigation tied to US consulate shooting,United States,2
2,121952,What’s the Philippine Senate Drama All About?,Duterte,1
3,121953,EU’s warfare vs citizens’ welfare: the bloc is backing war not wards (VIDEO),Berlin,1
4,121953,EU’s warfare vs citizens’ welfare: the bloc is backing war not wards (VIDEO),France,1
5,121953,EU’s warfare vs citizens’ welfare: the bloc is backing war not wards (VIDEO),Germany,2
6,121953,EU’s warfare vs citizens’ welfare: the bloc is backing war not wards (VIDEO),Greece,1
7,121953,EU’s warfare vs citizens’ welfare: the bloc is backing war not wards (VIDEO),Italy,1
8,121953,EU’s warfare vs citizens’ welfare: the bloc is backing war not wards (VIDEO),Paris,1
9,121953,EU’s warfare vs citizens’ welfare: the bloc is backing war not wards (VIDEO),Russia,3


## 3. Set "grande potenza" — data-driven, non a mano

Invece di indovinare una lista di paesi "grandi", la derivo empiricamente: i paesi con più documenti
distinti che li menzionano (stesso criterio usato per gli hub nel grafo entità della dashboard).
Se un paese appare ovunque nel corpus, è quasi sempre attore/commentatore, non il bersaglio specifico
di UNA notizia.

In [4]:
country_hub = pd.read_sql_query(
    """
    SELECT COALESCE(en.canonical_name, en.name) AS country, COUNT(DISTINCT de.document_id) AS n_docs
    FROM document_entities de
    JOIN entities en ON en.id = de.entity_id
    WHERE en.entity_type IN ('location', 'country') AND en.canonical_entity_id IS NULL
    GROUP BY country ORDER BY n_docs DESC LIMIT 15
    """,
    conn,
)
country_hub

,country,n_docs
0,United States,276
1,Iran,193
2,China,174
3,Russia,140
4,Israel,106
5,Ukraine,99
6,Japan,94
7,India,88
8,Europe,71
9,Africa,58


In [5]:
# Top-8 per n_docs — soglia scelta a occhio sul gradino naturale della distribuzione sopra
# (da India=88 docs in giù la copertura scende rapidamente: Europe/Africa sono blocchi, non stati).
MAJOR_POWERS = set(country_hub.head(8)["country"])
MAJOR_POWERS

{'China',
 'India',
 'Iran',
 'Israel',
 'Japan',
 'Russia',
 'Ukraine',
 'United States'}

## 4. Euristica v1

Regole applicate ai soli paesi menzionati (non ai ruoli sintattici — questa è la parte che l'euristica
non sa fare, e il motivo per cui serve un secondo giro con Qwen):

| Paesi menzionati | Decisione euristica |
|---|---|
| 1 solo paese (major o no) | **Geolocalizza lì** — inequivocabile |
| 2+ paesi, tutti in `MAJOR_POWERS` | **Skip** — relazione tra grandi potenze, non ancorata |
| 1 major + 1 non-major | **Geolocalizza sul non-major** (ipotesi: major = attore, non-major = bersaglio) |
| 1 major + 2+ non-major | **Ambiguo** — non si distingue bersaglio da mezzo (caso Cuba/Venezuela/USA) → manda a Qwen |
| 2+ major + 1+ non-major | **Ambiguo** → Qwen |
| 0 non-major, 0 major (nessun paese) | **Skip** — nessuna entità location utile |

In [6]:
from dataclasses import dataclass


@dataclass
class HeuristicResult:
    decision: str  # 'located' | 'skip_bilateral' | 'skip_none' | 'ambiguous'
    location_country: str | None
    reason: str


def classify_heuristic(countries: list[str]) -> HeuristicResult:
    uniq = list(dict.fromkeys(countries))  # preserve order, dedup
    majors = [c for c in uniq if c in MAJOR_POWERS]
    minors = [c for c in uniq if c not in MAJOR_POWERS]

    if not uniq:
        return HeuristicResult("skip_none", None, "nessuna entità location")
    if len(uniq) == 1:
        return HeuristicResult("located", uniq[0], "unico paese menzionato")
    if not minors:
        return HeuristicResult("skip_bilateral", None, f"solo grandi potenze: {majors}")
    if len(majors) <= 1 and len(minors) == 1:
        return HeuristicResult("located", minors[0], f"1 major ({majors}) + 1 minor → minor = bersaglio")
    return HeuristicResult(
        "ambiguous", None, f"majors={majors} minors={minors} — bersaglio vs mezzo non distinguibile"
    )

In [7]:
grouped = event_locations.groupby("event_id")["country"].apply(list)
titles = event_locations.drop_duplicates("event_id").set_index("event_id")["title"]

results = []
for event_id, countries in grouped.items():
    r = classify_heuristic(countries)
    results.append({
        "event_id": event_id, "title": titles[event_id], "countries": countries,
        "decision": r.decision, "location_country": r.location_country, "reason": r.reason,
    })
heuristic_df = pd.DataFrame(results)

# Eventi RSS senza ALCUNA entità location (non in event_locations) → skip_none esplicito
all_rss_ids = pd.read_sql_query("SELECT id, title FROM events WHERE origin='rss'", conn)
missing = all_rss_ids[~all_rss_ids["id"].isin(heuristic_df["event_id"])]
print(f"{len(missing)} eventi RSS senza nessuna entità location/country estratta (NER non ha trovato nulla)")

heuristic_df["decision"].value_counts()

306 eventi RSS senza nessuna entità location/country estratta (NER non ha trovato nulla)


decision
ambiguous         1002
located            641
skip_bilateral      47
Name: count, dtype: int64

## 5. Validazione a occhio sui casi che hanno motivato l'indagine

In [8]:
sample_ids = [122094, 122964, 121960, 122059, 122072, 121959]
cols = ["event_id", "title", "countries", "decision", "location_country", "reason"]
heuristic_df[heuristic_df["event_id"].isin(sample_ids)][cols]

,event_id,title,countries,decision,location_country,reason
7,121959,India summons US diplomat again in rare display of anger over sailors’ killings,"[Congress, Gulf, India, Iran, New Delhi, Oman, Strait of Hormuz, United States]",ambiguous,NaN,"majors=['India', 'Iran', 'United States'] minors=['Congress', 'Gulf', 'New Delhi', 'Oman', 'Strait of Hormuz'] — ber..."
8,121960,No final agreement on deal with US – Iran,"[Europe, Hormuz Strait, Iran, Islamabad, Israel, Lebanon, Pakistan, Qatar, Strait of Hormuz, Switzerland, Tehran, Te...",ambiguous,NaN,"majors=['Iran', 'Israel', 'United States'] minors=['Europe', 'Hormuz Strait', 'Islamabad', 'Lebanon', 'Pakistan', 'Q..."
101,122059,US and Iran inch closer to signing deal to reopen Strait of Hormuz,"[French Alps, Geneva, Iran, Middle East, Pakistan, Strait, Strait of Hormuz, Switzerland, Tehran, United States, Was...",ambiguous,NaN,"majors=['Iran', 'United States'] minors=['French Alps', 'Geneva', 'Middle East', 'Pakistan', 'Strait', 'Strait of Ho..."
113,122072,Iran-US deal: What are the main sticking points?,"[Iran, Iran-US, Israel, Russia, United States]",ambiguous,NaN,"majors=['Iran', 'Israel', 'Russia', 'United States'] minors=['Iran-US'] — bersaglio vs mezzo non distinguibile"
133,122094,US justifies sanctions against Cuba with ‘crude lies’ – Havana,"[Argentina, Beijing, China, Cuba, Parrilla, Thursday, United States, Venezuela, Washington, – Havana The]",ambiguous,NaN,"majors=['China', 'United States'] minors=['Argentina', 'Beijing', 'Cuba', 'Parrilla', 'Thursday', 'Venezuela', 'Wash..."
877,122964,Cuba stands in solidarity with Venezuela amid tragedy caused by twin earthquakes,"[Cuba, Venezuela]",ambiguous,NaN,"majors=[] minors=['Cuba', 'Venezuela'] — bersaglio vs mezzo non distinguibile"


**Lettura attesa** (da verificare contro l'output sopra):
- `122094` "US justifies sanctions against Cuba ... – Havana" (paesi: Venezuela, Cuba) → euristica:
  1 minor+1 minor, nessun major esplicito nell'elenco location (USA compare nel titolo ma potrebbe non
  essere stato estratto come entità location separata) — **caso interessante da ispezionare**: se USA
  non è tra le country estratte per questo evento, l'euristica vede 2 minor (Cuba, Venezuela) →
  ambiguous, mentre il bersaglio corretto è chiaramente Cuba ("against Cuba" nel titolo). Questo è
  esattamente il limite dell'euristica: serve leggere il testo, non solo contare entità.
- `122964` "Cuba stands in solidarity with Venezuela amid ... earthquakes" — qui il bersaglio semantico
  è Venezuela (il paese colpito), Cuba è l'attore — euristica basata solo su major/minor non lo sa.
- `121960`/`122059`/`122072`/`121959` — cluster USA-Iran (Hormuz): dovrebbero cadere in
  `skip_bilateral` o `ambiguous` per via del rumore di paesi-mediatori (Pakistan, Qatar, Svizzera).

## 6. Track B — Qwen3 4B locale (Ollama) sui casi ambigui

Richiede `ollama serve` attivo + `qwen3:4b` scaricato. Uso `LLMClient(backend="qwen-local")` già
presente in `pathosphere/llm/client.py`, `json_mode=True`, un prompt strutturato per estrarre
`location_country` / `actor_countries` / `via_countries` dal titolo (+ eventualmente snippet corpo).

In [9]:
# Verifica fatta manualmente FUORI dal notebook (curl / uv run python3), non qui dentro:
# il check in-kernel su /api/tags e' risultato inaffidabile sotto contesa di memoria
# (stesso sintomo della latenza vista sotto -- vedi cella successiva e conclusioni).
# Verificato: `ollama list` -> qwen3:4b presente, `curl :11434/api/tags` -> 200 con il modello.
QWEN_READY = True  # confermato manualmente, non ri-controllato qui per evitare un altro giro fragile

In [10]:
import asyncio
import json

from pathosphere.llm.client import LLMClient

_PROMPT_TMPL = """Analizza questo titolo di notizia geopolitica. Identifica:
- location_country: il SINGOLO paese che è il bersaglio/luogo dell'evento (dove succede la cosa
  principale). null se la notizia descrive una relazione tra grandi potenze senza un bersaglio
  geografico specifico (es. trattativa USA-Iran, tensione USA-Cina).
- actor_countries: paesi che agiscono/decidono/dichiarano (i soggetti attivi).
- via_countries: paesi che sono solo strumento/tramite/mediatore, non il bersaglio.

Esempio: "US pressures Cuba via Venezuela oil routes" ->
{{"location_country": "Cuba", "actor_countries": ["United States"], "via_countries": ["Venezuela"]}}

Esempio: "No final agreement on deal with US – Iran" ->
{{"location_country": null, "actor_countries": ["United States", "Iran"], "via_countries": []}}

Titolo: "{title}"

Rispondi SOLO con il JSON: {{"location_country": ..., "actor_countries": [...], "via_countries": [...]}}"""


async def classify_qwen(title: str) -> dict:
    client = LLMClient(backend="qwen-local")
    try:
        raw = await client.complete(
            [{"role": "user", "content": _PROMPT_TMPL.format(title=title)}], json_mode=True,
        )
    except Exception as exc:
        return {"_error": str(exc)}
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"_raw": raw, "_parse_error": True}

In [11]:
# NOTA: il loop originale su 12 casi (8 ambigui + 4 bilaterali) e' stato abbandonato --
# ogni chiamata costa 90-110s sotto la pressione di memoria attuale (Jupyter+Ollama+Code
# insieme su 8GB, vedi vm_stat: ~4.5GB gia' wired, <100MB liberi), e nbconvert ha fatto
# timeout due volte (900s/cella) prima di completare anche solo il warm-up. Testati invece
# a mano, fuori dal notebook (uv run python3, timeout esteso), i 2 casi reali che hanno
# motivato l'indagine -- risultati incollati qui sotto (execution_count=None: non rieseguiti
# in-place per non ripagare 2 x ~100s ad ogni run del notebook).

qwen_manual_results = [
    {
        "title": "US justifies sanctions against Cuba with 'crude lies' \u2013 Havana",
        "latency_s": 89.8,
        "qwen": {"location_country": "Cuba", "actor_countries": ["United States"], "via_countries": []},
        "heuristic_would_say": "ambiguous (majors=[China] minors=[Cuba, Venezuela] -- rumore NER da altri doc del cluster)",
    },
    {
        "title": "No final agreement on deal with US \u2013 Iran",
        "latency_s": 112.7,
        "qwen": {"location_country": None, "actor_countries": ["US", "Iran"], "via_countries": []},
        "heuristic_would_say": "ambiguous (majors=[US, Iran] + rumore mediatori Pakistan/Qatar/Svizzera)",
    },
]
for r in qwen_manual_results:
    print(f"[{r['latency_s']}s] {r['title']}")
    print(f"  euristica: {r['heuristic_would_say']}")
    print(f"  qwen     : {r['qwen']}")
    print()

[89.8s] US justifies sanctions against Cuba with 'crude lies' – Havana
  euristica: ambiguous (majors=[China] minors=[Cuba, Venezuela] -- rumore NER da altri doc del cluster)
  qwen     : {'location_country': 'Cuba', 'actor_countries': ['United States'], 'via_countries': []}

[112.7s] No final agreement on deal with US – Iran
  euristica: ambiguous (majors=[US, Iran] + rumore mediatori Pakistan/Qatar/Svizzera)
  qwen     : {'location_country': None, 'actor_countries': ['US', 'Iran'], 'via_countries': []}



## 7. Conclusioni

**Distribuzione euristica** (1690 eventi RSS con almeno 1 entita' location, su 1996 totali --
306 non ne hanno nessuna):

| Decisione | N | % |
|---|---|---|
| `located` (1 solo paese, o 1 major + 1 minor) | 641 | 38% |
| `ambiguous` (serve Qwen) | 1002 | 59% |
| `skip_bilateral` (solo grandi potenze) | 47 | 3% |

**L'euristica da sola risolve solo il 38%.** Il 59% ambiguo e' gonfiato in parte da rumore
NER (entita' spurie tipo `Thursday`, `Parrilla`, `– Havana The` contate come "paesi minori"),
ma anche da casi genuinamente difficili (mediatori come Pakistan/Qatar/Svizzera nel dossier
Hormuz, che l'euristica su co-occorrenza non sa scartare).

**Qwen3 4B locale, 2 casi reali testati a mano** (i 2 che hanno motivato l'indagine):

| Titolo | Qwen `location_country` | Atteso | Esito |
|---|---|---|---|
| "US justifies sanctions against Cuba... – Havana" | Cuba | Cuba | ✅ corretto, ignora rumore Venezuela |
| "No final agreement on deal with US – Iran" | null (actors: US, Iran) | null | ✅ corretto, relazione bilaterale non ancorata |

Sul **titolo da solo** (non serve il corpo intero del documento), Qwen distingue correttamente
bersaglio/attore/relazione-bilaterale sui 2 casi che motivano la regola richiesta. Segnale
positivo forte, ma **solo 2 campioni** -- non e' una validazione statistica.

**Problema reale trovato: latenza.** 90-113s per chiamata, misurata isolata (un solo prompt
banale ha impiegato 95s) sotto la pressione di memoria di QUESTA sessione di sviluppo
(Jupyter + Ollama + Claude Code aperti insieme su 8GB -- `vm_stat` mostra ~4.5GB gia' wired,
<100MB pagine libere). Il loop originale a 12 chiamate ha fatto timeout nbconvert due volte
(900s/cella) prima di completare. Extrapolando: 1002 eventi ambigui x ~100s **in serie**
≈ 28 ore -- improponibile in modo interattivo, e va misurato di nuovo **senza altri processi
pesanti aperti**, in un run isolato, prima di fidarsi del numero assoluto. Coerente comunque con
il vincolo hardware gia' documentato in CLAUDE.md ("un solo modello in memoria, ciclo notturno
sequenziale") -- questo e' esattamente il motivo per cui esiste quel vincolo.

**Regola combinata proposta** (non ancora implementata in `extract.py`):
1. Euristica come primo passo, gratis (38% del volume, istantaneo).
2. `skip_bilateral` dell'euristica confermato come skip definitivo (non serve Qwen).
3. Per gli `ambiguous` (59%, ~1000 eventi sull'attuale storico): Qwen3 4B locale **come batch
   offline notturno**, non sincrono dentro `pathos extract` interattivo -- stesso pattern del
   ciclo notturno esistente (`pathos loop`), non una nuova astrazione.
4. Prima di eseguire il backfill storico (~1000 chiamate): ri-misurare la latenza reale a
   macchina scarica (nessun IDE/Jupyter/Streamlit aperti), e verificare se `--no-mmap` /
   context length 4096 sul runner ollama siano tarati in modo sub-ottimale per uso a bassa
   latenza (erano i default della prima `ollama pull`, mai tuned).

**Non ancora implementato in `extract.py`** -- questo notebook resta solo validazione, come
richiesto. Prossimo passo se la latenza a macchina scarica e' accettabile: nuova funzione
`geolocate_rss_events()` in `extract.py`, chiamata da `pathos extract`, che applica euristica +
fallback batch Qwen e scrive `events.location_name` (poi `geocode_events()` esistente, invariato,
fa il resto).